In [6]:
import json
import numpy as np
from google.cloud import aiplatform

# ── Initialise Vertex AI ──────────────────────────────────────────────────────
aiplatform.init(
    project  = "YOUR_GCP_PROJECT_ID",
    location = "YOUR_REGION"
)

# ── Connect to both live endpoints ───────────────────────────────────────────
uk_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/6178114055031488512"
)
sea_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/640375363226042368"
)

# ── Load feature lists ────────────────────────────────────────────────────────
uk_feature_cols   = json.load(open("/home/jupyter/models/feature_cols_uk.json"))
sea_feature_cols = json.load(open("/home/jupyter/models/feature_cols_sea.json"))

print(f"UK endpoint   : {uk_endpoint.resource_name}")
print(f"Sea endpoint : {sea_endpoint.resource_name}")
print(f"UK features   : {len(uk_feature_cols)}")
print(f"Sea features : {len(sea_feature_cols)}")
print("\n✅ Ready to predict")

UK endpoint   : projects/128825737789/locations/YOUR_REGION/endpoints/6178114055031488512
Sea endpoint : projects/128825737789/locations/YOUR_REGION/endpoints/640375363226042368
UK features   : 97
Sea features : 78

✅ Ready to predict


In [7]:
def build_payload(feature_cols, values_dict):
    payload = [0.0] * len(feature_cols)
    for feat, val in values_dict.items():
        if feat in feature_cols:
            payload[feature_cols.index(feat)] = float(val)
    return payload

# ── Dove Body Wash — Tesco TPR — Feb 2024 ────────────────────────────────────
uk_values = {
    "Level8Code"                         : 350300,
    "WeekSkID"                           : 6,
    "InStoreStartWeek"                   : 2024.06,
    "InStoreEndWeek"                     : 2024.08,
    "ListPrice"                          : 3.50,
    "PlannedPromoSalesVolumeSellIn"       : 3200,
    "PlannedNetPromoGSVSellIn"           : 11200,
    "PlannedTTSOnSpend"                  : 3200,
    "PlannedNetPromoNIVSellIn"           : 8000,
    "PlannedTTSOffSpend"                 : 1600,
    "PlannedNetPromoTOSellIn"            : 6400,
    "PlannedNetPromoGrossProfitsSellIn"  : 8200,
    "PlannedNetPromoCOGSSellIn"          : 5800,
    "PlannedBaselineVolume"              : 1100,
    "PlannedBaseGSVSellIn"               : 3850,
    "PlannedBaseTTSOnSpend"              : 900,
    "PlannedBaseNIVSellIn"               : 2950,
    "PlannedBaseTTSOffSpend"             : 550,
    "PlannedBaseTOSellIn"                : 2400,
    "PlannedBaseGrossProfitsSellIn"      : 2800,
    "PlannedBaseCOGSSellIn"              : 2000,
    "PlannedEventSpend"                  : 1200,
    "IsPipeFill"                         : 0,
    "IsDefensivePromo"                   : 0,
    "PlannedIncrementalVolume"           : 2100,
    "PlannedUpliftRate"                  : 1.91,
    "PlannedTTSTotal"                    : 4800,
    "PlannedCostPerUnit"                 : 1.50,
    "PlannedROI_proxy"                   : 0.71,
    "PromoDurationWeeks"                 : 2,
    "InstoreStartDate_Month"             : 2,
    "InstoreStartDate_Week"              : 6,
    "InstoreEndDate_Month"               : 2,
    "InstoreEndDate_Week"                : 8,
    "ShipmentStartDate_Month"            : 1,
    "ShipmentStartDate_Week"             : 4,
    "ShipmentEndDate_Month"              : 2,
    "ShipmentEndDate_Week"               : 6,
    "Customer_TESCO STORES LTD"          : 1,
    "PromotionStatus_Executed"           : 1,
    "DivisionName_VG_HPC CATEGORY"       : 1,
    "PromoMechanic_TPR"                  : 1,
    "PromoFeature_Gondola End"           : 1,
    "CategoryName_VG_SKIN CLEANSING"     : 1,
}

payload  = build_payload(uk_feature_cols, uk_values)
response = uk_endpoint.predict(instances=[payload])
pred_log   = response.predictions[0]
pred_units = max(0, round(np.expm1(pred_log)))

planned      = 3200
spend        = 4800
gp_per_unit  = 2.56
variance_pct = ((pred_units - planned) / planned) * 100
roi_planned  = (planned    * gp_per_unit - spend) / spend
roi_model    = (pred_units * gp_per_unit - spend) / spend
extra_profit = round(pred_units * gp_per_unit - spend)

print("=" * 60)
print("  UNILEVER PROMO INTELLIGENCE — VERTEX AI LIVE PREDICTION")
print("=" * 60)
print(f"\n  Customer   : Tesco Stores Ltd")
print(f"  Brand      : Dove Body Wash — Skin Cleansing")
print(f"  Mechanic   : Temporary Price Reduction (TPR)")
print(f"  Placement  : Gondola End — Feb 2024 (2 weeks)")
print(f"\n  {'Planner estimate':<28} {planned:>8,} units")
print(f"  {'Model prediction':<28} {pred_units:>8,} units")
print(f"  {'Variance':<28} {variance_pct:>+7.1f}%")
print(f"\n  {'Trade spend paid to Tesco':<28} £{spend:>7,}")
print(f"  {'Planner expected return':<28} {roi_planned:>+7.2f}x")
print(f"  {'Model-adjusted return':<28} {roi_model:>+7.2f}x")
print(f"  {'Extra profit for FMCG Client':<28} £{extra_profit:>+7,}")
print("=" * 60)

  UNILEVER PROMO INTELLIGENCE — VERTEX AI LIVE PREDICTION

  Customer   : Tesco Stores Ltd
  Brand      : Dove Body Wash — Skin Cleansing
  Mechanic   : Temporary Price Reduction (TPR)
  Placement  : Gondola End — Feb 2024 (2 weeks)

  Planner estimate                3,200 units
  Model prediction                    0 units
  Variance                      -100.0%

  Trade spend paid to Tesco    £  4,800
  Planner expected return        +0.71x
  Model-adjusted return          -1.00x
  Extra profit for FMCG Client    £ -4,800


In [8]:
import pandas as pd

eval_df = pd.read_csv("/home/jupyter/models/uk_eval_results.csv")
print(f"Rows: {eval_df.shape[0]:,}")
print(f"Columns: {eval_df.shape[1]}")
print(eval_df.head(2))

Rows: 46,644
Columns: 108
   Level8Code          TUEAN  WeekSkID  InStoreStartWeek  InStoreEndWeek  \
0      335871  8711327534339        79           2023.02         2023.05   
1      335871  8711327534339        78           2023.02         2023.05   

   PromoShopperMechanic  SegmentName_VG  FormName_VG  EBFName_VG  Brand_VG  \
0                    36             100           86          32         8   
1                    36             100           86          32         8   

   ...  PredictedSellOut  PredictedVolumeRatio  PredictedPromoGP  \
0  ...          649.1471              0.236578        291.526747   
1  ...          341.5662              0.124482        153.394634   

   PredictedIncrementalGP  TotalSpend  PredictedROI  PlannedROI  \
0            -1120.749163  2282.67446     -0.490981    -0.07886   
1            -1258.881276  2282.67446     -0.551494    -0.07886   

   ModelAccuracyRatio  AdjustedIncrementalGP  AdjustedROI  
0            0.953226            -171.59156

In [9]:
# ── Extract only the 97 feature columns from a real row ──────────────────────
real_row = eval_df[uk_feature_cols].iloc[0]
payload  = real_row.fillna(0).tolist()

# ── Call the endpoint ─────────────────────────────────────────────────────────
response   = uk_endpoint.predict(instances=[payload])
pred_log   = response.predictions[0]
pred_units = max(0, round(np.expm1(pred_log)))

# ── Get the actual values for comparison ─────────────────────────────────────
actual_units  = round(eval_df['ActualSellOut'].iloc[0])
planned_units = round(eval_df['PlannedPromoSalesVolumeSellIn'].iloc[0])
spend         = round(eval_df['TotalSpend'].iloc[0])
gp_per_unit   = 0.45   # conservative GP per unit from eval
variance_pct  = ((pred_units - actual_units) / actual_units * 100) if actual_units else 0
roi_model     = (pred_units * gp_per_unit - spend) / spend if spend else 0
roi_planned   = eval_df['PlannedROI'].iloc[0]

print("=" * 60)
print("  UNILEVER PROMO INTELLIGENCE — VERTEX AI LIVE PREDICTION")
print("=" * 60)
print(f"\n  Using real promotion row from UK test set")
print(f"  Level8Code  : {int(eval_df['Level8Code'].iloc[0])}")
print(f"  TUEAN       : {int(eval_df['TUEAN'].iloc[0])}")
print(f"  Week        : {eval_df['WeekSkID'].iloc[0]}")
print(f"\n  {'Actual sell-out':<28} {actual_units:>8,} units")
print(f"  {'Planner estimate':<28} {planned_units:>8,} units")
print(f"  {'Model prediction':<28} {pred_units:>8,} units")
print(f"  {'Model vs actual':<28} {variance_pct:>+7.1f}%")
print(f"\n  {'Trade spend':<28} £{spend:>7,}")
print(f"  {'Planner expected return':<28} {roi_planned:>+7.3f}x")
print(f"  {'Model-adjusted return':<28} {roi_model:>+7.2f}x")
print("=" * 60)

  UNILEVER PROMO INTELLIGENCE — VERTEX AI LIVE PREDICTION

  Using real promotion row from UK test set
  Level8Code  : 335871
  TUEAN       : 8711327534339
  Week        : 79

  Actual sell-out                   681 units
  Planner estimate                2,744 units
  Model prediction                    2 units
  Model vs actual                -99.7%

  Trade spend                  £  2,283
  Planner expected return       -0.079x
  Model-adjusted return          -1.00x


In [10]:
response = uk_endpoint.predict(instances=[payload])
raw = response.predictions[0]

print(f"Raw response type : {type(raw)}")
print(f"Raw response value: {raw}")
print(f"After expm1       : {np.expm1(raw)}")
print(f"Payload length    : {len(payload)}")
print(f"First 5 values    : {payload[:5]}")

Raw response type : <class 'float'>
Raw response value: 1.243443965911865
After expm1       : 2.467534996631246
Payload length    : 97
First 5 values    : [335871.0, 8711327534339.0, 79.0, 2023.02, 2023.05]


In [12]:
import subprocess
subprocess.run(['gsutil', 'cp', 'gs://YOUR_BUCKET_NAME/models/xgb_uk_tuned.json', '/home/jupyter/models/xgb_uk_tuned.json'])
print("Done")

Copying gs://YOUR_BUCKET_NAME/models/xgb_uk_tuned.json...
/ [1 files][ 10.1 MiB/ 10.1 MiB]                                                
Operation completed over 1 objects/10.1 MiB.                                     


Done


In [13]:
import xgboost as xgb

booster = xgb.Booster()
booster.load_model("/home/jupyter/models/xgb_uk_tuned.json")

booster_cols = booster.feature_names

print("Match?", booster_cols == uk_feature_cols)
print("\nFirst 5 booster cols:", booster_cols[:5])
print("First 5 payload cols:", uk_feature_cols[:5])

# Find mismatches
mismatches = [(i, b, u) for i, (b, u) in enumerate(zip(booster_cols, uk_feature_cols)) if b != u]
print(f"\nMismatched positions: {len(mismatches)}")
for i, b, u in mismatches[:5]:
    print(f"  pos {i}: booster='{b}' vs feature_cols='{u}'")

Match? True

First 5 booster cols: ['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek']
First 5 payload cols: ['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek']

Mismatched positions: 0


In [14]:
import pandas as pd, xgboost as xgb

real_row = eval_df[booster_cols].iloc[0]
X = pd.DataFrame([real_row.fillna(0)], columns=booster_cols).astype(float)
dmat = xgb.DMatrix(X, feature_names=booster_cols)
log_pred = booster.predict(dmat)[0]

print(f"Local booster  : {np.expm1(log_pred):,.0f} units")
print(f"Vertex endpoint: 2 units")
print(f"Actual         : 681 units")

Local booster  : 649 units
Vertex endpoint: 2 units
Actual         : 681 units


In [15]:
response = uk_endpoint.predict(instances=[payload])
raw = response.predictions[0]
pred_units = max(0, round(np.expm1(raw)))

print(f"Raw response  : {raw}")
print(f"Predicted     : {pred_units:,} units")
print(f"Actual        : 681 units")
print(f"Local booster : 649 units")

InternalServerError: 500 <!doctype html>
<html lang=en>
<title>500 Internal Server Error</title>
<h1>Internal Server Error</h1>
<p>The server encountered an internal error and was unable to complete your request. Either the server is overloaded or there is an error in the application.</p>


In [17]:
response = uk_endpoint.predict(instances=[payload])
raw = response.predictions[0]
pred_units = max(0, round(np.expm1(raw)))

print(f"Raw response  : {raw}")
print(f"Predicted     : {pred_units:,} units")
print(f"Actual        : 681 units")
print(f"Local booster : 649 units")

Raw response  : 6.477198600769043
Predicted     : 649 units
Actual        : 681 units
Local booster : 649 units


In [18]:
from google.cloud import aiplatform
import numpy as np, json, pandas as pd

aiplatform.init(project="YOUR_GCP_PROJECT_ID", location="YOUR_REGION")

sea_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/640375363226042368"
)

# Load feature list
sea_feature_cols = json.load(open('/home/jupyter/models/feature_cols_sea.json'))

# Load eval data to get a real row
eval_df = pd.read_csv('/home/jupyter/models/sea_eval_results.csv')

payload = eval_df[sea_feature_cols].iloc[0].fillna(0).tolist()

# Call endpoint
response = sea_endpoint.predict(instances=[payload])
raw = response.predictions[0]
pred_units = max(0, round(np.expm1(raw)))

print(f'Raw (log-scale) : {raw:.3f}')
print(f'Predicted       : {pred_units:,} units')

FileNotFoundError: [Errno 2] No such file or directory: '/home/jupyter/models/indo_eval_results.csv'

In [19]:
from google.cloud import aiplatform
import numpy as np, json, pandas as pd

aiplatform.init(project="YOUR_GCP_PROJECT_ID", location="YOUR_REGION")

sea_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/640375363226042368"
)

# Load feature list
sea_feature_cols = json.load(open('/home/jupyter/models/feature_cols_sea.json'))

# Build a zero-filled synthetic row (safe baseline test)
payload = [0.0] * len(sea_feature_cols)

# Call endpoint
response = sea_endpoint.predict(instances=[payload])
raw = response.predictions[0]
pred_units = max(0, round(np.expm1(raw)))

print(f'Feature count   : {len(sea_feature_cols)} features')
print(f'Raw (log-scale) : {raw:.3f}')
print(f'Predicted       : {pred_units:,} units')

Feature count   : 78 features
Raw (log-scale) : -0.583
Predicted       : 0 units


In [20]:
from google.cloud import aiplatform
import numpy as np, json, pandas as pd

aiplatform.init(project="YOUR_GCP_PROJECT_ID", location="YOUR_REGION")

sea_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/640375363226042368"
)

# Load feature list
sea_feature_cols = json.load(open('/home/jupyter/models/feature_cols_sea.json'))

# Build a zero-filled synthetic row (safe baseline test)
payload = [0.0] * len(sea_feature_cols)

# Call endpoint
response = sea_endpoint.predict(instances=[payload])
raw = response.predictions[0]
pred_units = max(0, round(np.expm1(raw)))

print(f'Feature count   : {len(sea_feature_cols)} features')
print(f'Raw (log-scale) : {raw:.3f}')
print(f'Predicted       : {pred_units:,} units')

Feature count   : 78 features
Raw (log-scale) : -0.583
Predicted       : 0 units
